In [1]:
from sklearn.metrics import mean_absolute_error
from keras.layers import Dense, Dropout
from keras.models import Sequential

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn import linear_model
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from math import sqrt
df = pd.read_excel("../../clean_coast_data.xlsx")
df.dropna(inplace=True)
df = df.loc[df.winter == 0]

In [2]:
# using get_dummies on education column
encoded = pd.get_dummies(df.Area)
# Concatenate the dummies to original dataframe
merged_data = pd.concat([df, encoded], axis=1)
# dropping the original column which was not encoded
merged_data.drop(columns='Area', inplace=True)

# merged_data.drop(columns=['coast_name','coast_desc','city','check station','key','coast_num','center'], inplace=True)
merged_data['Date test num'] = merged_data['Date test'].view(np.int64)

import sys
sys.path.append('../')
import utils
# print(without_nulls.columns)
merged_data = utils.remove_if_exist(merged_data)

merged_data = utils.remove_high_corr(merged_data)

In [3]:
X = merged_data.drop(columns=['test_result','test_result_num'])
y = merged_data[['test_result']]

In [4]:
X.drop('Date test',axis=1, inplace=True)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=312148513)

X_train_scaler = MinMaxScaler()
X_test_scaler = MinMaxScaler()

X_train_scaled = X_train_scaler.fit_transform(X_train)
X_test_scaled = X_test_scaler.fit_transform(X_test)

# y_train_scaled = np.log(y_train+0.001)
# y_test_scaled = np.log(y_test+0.001)

dataset_train = pd.merge(left=X_train, right=y_train, left_index=True, right_index=True)
dataset_test = pd.merge(left=X_test, right=y_test, left_index=True, right_index=True)

In [5]:
from keras import backend
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from IPython.display import clear_output
from keras.regularizers import l2
from sklearn.metrics import r2_score
stds_train = list()
rmses_train = list()
r2s_train = list()

stds_test = list()
rmses_test = list()
r2s_test = list()
# create NN model
model = Sequential()
 
model.add(Dense(X_train.shape[1],input_dim=X_train.shape[1],activation='sigmoid'))

model.add(Dense(64,activation='sigmoid'))
model.add(Dropout(0.5))
model.add(Dense(32,activation='sigmoid'))
model.add(Dropout(0.5))
model.add(Dense(4,activation='sigmoid'))
model.add(Dropout(0.5))
model.add(Dense(1))
 
def get_rmse(x,y):
  return sqrt(mean_squared_error(x,y))

for i in range(100):
    # Compiling the model
    model.compile(metrics=['mae','accuracy'], optimizer='adam', loss = 'mean_squared_error')
    es = EarlyStopping(monitor='val_loss', min_delta=1e-10, patience=25, verbose=1 )
    # rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=0 )
    # mcp = ModelCheckpoint(filepath='weights_Ramon.h5', monitor='val_loss', verbose=1 , save_best_only=True, save_weights_only=True)

    # tb = TensorBoard('logs')

    history = model.fit(X_train_scaled, y_train, shuffle=True, epochs=50 ,validation_split=0.3,batch_size=32, callbacks=[es])



    #Train
    prediction =model.predict(X_train_scaled)
    # print('')
    resTrain = pd.DataFrame(
        data={
            'Prediction':prediction.T[0],
            'Actual': y_train.test_result.values
        },
        index=y_train.index
    )

    #Test
    prediction =model.predict(X_test_scaled)
    # print('')
    resTest = pd.DataFrame(
        data={
            'Prediction':prediction.T[0],
            'Actual': y_test.test_result.values
        },
        index=y_test.index
    )

    # print('rmse train',sqrt(mean_squared_error(resTrain.Actual, resTrain.Prediction)))
    # print('std train',np.std(resTrain.Actual))
    # print('r2 train',r2_score(resTrain.Actual, resTrain.Prediction))
    # print('')

    # print('rmse test',sqrt(mean_squared_error(resTest.Actual, resTest.Prediction)))
    # print('std test',np.std(resTest.Actual))
    # print('r2 test',r2_score(resTest.Actual, resTest.Prediction))
    train_rmse = sqrt(mean_squared_error(resTrain.Prediction, resTrain.Actual))
    train_std = np.std(resTrain.Actual)
    train_r2 = r2_score(resTrain.Actual, resTrain.Prediction)
    test_rmse = sqrt(mean_squared_error(resTest.Prediction, resTest.Actual))
    test_std = np.std(resTest.Actual)
    test_r2 = r2_score(resTest.Actual, resTest.Prediction)
    stds_train.append(train_std)
    rmses_train.append(train_rmse)
    r2s_train.append(train_r2)
    stds_test.append(test_std)
    rmses_test.append(test_rmse)
    r2s_test.append(test_r2)

print("train std: ",np.mean(stds_train))
print("train rmse: ",np.mean(rmses_train))
print("train r2: ",np.mean(r2s_train))
print()
print("test std: ",np.mean(stds_test))
print("test rmse: ",np.mean(rmses_test))
print("test r2: ",np.mean(r2s_test))

Epoch 1/50
291/291 [==============================] - 3s 6ms/step - loss: 12.0267 - mae: 3.2879 - accuracy: 0.0086 - val_loss: 9.4942 - val_mae: 2.9218 - val_accuracy: 0.0213
Epoch 2/50
291/291 [==============================] - 1s 3ms/step - loss: 7.5288 - mae: 2.4741 - accuracy: 0.0145 - val_loss: 4.9507 - val_mae: 2.0316 - val_accuracy: 0.0213
Epoch 3/50
291/291 [==============================] - 1s 3ms/step - loss: 4.9727 - mae: 1.8868 - accuracy: 0.0231 - val_loss: 3.0676 - val_mae: 1.5586 - val_accuracy: 0.0213
Epoch 4/50
291/291 [==============================] - 1s 3ms/step - loss: 4.0621 - mae: 1.6581 - accuracy: 0.0231 - val_loss: 2.1814 - val_mae: 1.2794 - val_accuracy: 0.0213
Epoch 5/50
291/291 [==============================] - 1s 4ms/step - loss: 3.6232 - mae: 1.5613 - accuracy: 0.0231 - val_loss: 1.7866 - val_mae: 1.1216 - val_accuracy: 0.0213
Epoch 6/50
291/291 [==============================] - 1s 3ms/step - loss: 3.3967 - mae: 1.5001 - accuracy: 0.0231 - val_loss: 1.5